# Ain-Bondhu: Fine-tune Gemma 4 E4B → GGUF
Run this on **RunPod** (any GPU with 16GB+ VRAM).
Produces a GGUF file you can run on your local PC with llama.cpp.
No Hugging Face token needed — GGUF is saved locally.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import torch
import os, json, requests
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

## 1. Load Gemma 4 E4B (4-bit QLoRA)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4-E4B-it",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Load Training Data (Bangla Worker Rights)

In [ ]:
url = "https://raw.githubusercontent.com/hodini007/Ain-Bondhu/main/data/training/worker_rights_qa.json"
samples = requests.get(url).json()
print(f"Loaded {len(samples)} training samples")

In [ ]:
prompt_template = """<bos><start_of_turn>user
{instruction}<end_of_turn>
<start_of_turn>model
{output}<end_of_turn></bos>"""

texts = [prompt_template.format(**s) for s in samples]
dataset = Dataset.from_list([{"text": t} for t in texts])
print(f"Dataset ready: {len(dataset)} samples")

## 3. Fine-tune

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer_stats = trainer.train()
print(f"Training complete! Loss: {trainer_stats.training_loss:.4f}")

## 4. Merge LoRA → Full Model

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained("gemma4_bn_worker_merged")
tokenizer.save_pretrained("gemma4_bn_worker_merged")
print("Merged model saved to gemma4_bn_worker_merged/")

## 5. Convert to GGUF
Clones llama.cpp and converts the merged model to GGUF format (Q4_K_M quantization).

In [ ]:
!rm -rf llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -r llama.cpp/requirements.txt -q
!mkdir -p llama.cpp/build && cd llama.cpp/build && cmake .. && cmake --build . --config Release -j 2>&1 | tail -5

In [ ]:
# Use convert.py (latest llama.cpp) or fallback to convert_hf_to_gguf.py
import os
if os.path.exists("llama.cpp/convert.py"):
    !python llama.cpp/convert.py gemma4_bn_worker_merged --outfile gemma4_bn_worker_q4_k_m.gguf --outtype q4_k_m
elif os.path.exists("llama.cpp/convert_hf_to_gguf.py"):
    !python llama.cpp/convert_hf_to_gguf.py gemma4_bn_worker_merged --outfile gemma4_bn_worker_q4_k_m.gguf --outtype q4_k_m
else:
    print("ERROR: No conversion script found in llama.cpp")

In [ ]:
import os
size_mb = os.path.getsize("gemma4_bn_worker_q4_k_m.gguf") / 1024**2
print(f"✅ GGUF file size: {size_mb:.1f} MB")
print(f"📥 Download via RunPod File Manager → gemma4_bn_worker_q4_k_m.gguf")

## 6. Test Inference
Runs a quick test using the GGUF file via llama.cpp's built-in server.

In [ ]:
# Check if GGUF exists before starting server
import os
if not os.path.exists("gemma4_bn_worker_q4_k_m.gguf"):
    print("❌ GGUF not found. Run the conversion cell first.")
else:
    import subprocess
    server_path = "./llama.cpp/build/bin/llama-server"
    if not os.path.exists(server_path):
        server_path = "./llama.cpp/llama-server"
    if os.path.exists(server_path):
        !nohup {server_path} -m gemma4_bn_worker_q4_k_m.gguf --host 0.0.0.0 --port 8080 -ngl 0 --ctx-size 2048 > /dev/null 2>&1 &
        !sleep 3
        print("✅ llama-server started on port 8080")
    else:
        print("⚠️  llama-server binary not found. Test locally after download.")
        print("   Run on your PC: llama-server -m gemma4_bn_worker_q4_k_m.gguf --port 8080")

In [ ]:
import requests
try:
    resp = requests.post("http://localhost:8080/v1/chat/completions", json={
        "model": "gemma4_bn_worker_q4_k_m",
        "messages": [
            {"role": "user", "content": "আমার বেতন ৪ মাস বকেয়া, কী করব?"}
        ],
        "temperature": 0.4,
        "max_tokens": 256,
    }, timeout=30)
    print(resp.json()["choices"][0]["message"]["content"])
except Exception as e:
    print(f"❌ Cannot reach server: {e}")
    print("Test the GGUF locally after download:")
    print("  llama-server -m gemma4_bn_worker_q4_k_m.gguf -ngl 0 --host 0.0.0.0 --port 8080")

## Done!
Download `gemma4_bn_worker_q4_k_m.gguf` from RunPod File Manager.
On your local PC:
```
llama-server -m gemma4_bn_worker_q4_k_m.gguf -ngl 0 --host 0.0.0.0 --port 8080
```
Then in app settings ⚙️ → Base URL: `http://localhost:8080/v1`